In [32]:

import pandas as pd
import yfinance as yf
# Download historical closing prices for M7 + SPY
df=yf.download(["SPY","GOOGL", "AMZN", "AAPL", "META", "MSFT", "NVDA", "TSLA"],start="2012-05-18", end="2026-09-16")["Close"]
df.dropna(inplace=True)
df

[*********************100%***********************]  8 of 8 completed


Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA
Date,,,,,,,,
2012-05-18,15.863472,10.692500,14.883400,37.897205,23.079256,0.276238,101.239090,1.837333
2012-05-21,16.787676,10.905500,15.223258,33.733757,23.457737,0.281040,102.979134,1.918000
2012-05-22,16.658770,10.766500,14.893315,30.730141,23.465630,0.277610,103.158638,2.053333
2012-05-23,17.065237,10.864000,15.107990,31.721437,22.953108,0.284470,103.213257,2.068000
2012-05-24,16.908514,10.762000,14.964211,32.742466,22.921558,0.276924,103.416138,2.018667
...,...,...,...,...,...,...,...,...
2026-09-09,315.339996,252.399994,330.649994,653.690002,491.649994,223.419998,762.400024,367.809998
2026-09-10,326.570007,251.889999,332.600006,644.380005,492.440002,218.360001,757.830017,363.559998
2026-09-11,332.269989,256.779999,338.500000,648.030029,495.630005,218.289993,764.289978,365.440002


1st part

In [33]:
# Normalize to % return from starting date
normalized = ((df / df.iloc[0])-1)*100
# Equal weight M7 average (ideal/theoretical distribution — 1/7 each)
normalized["M7"]= normalized[["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"]].mean(axis=1)
# Simulating 1000 randomly weighted M7 portfolios and averaging to approximate realistic investor returns
import numpy as np
mag7 = ["GOOGL", "AMZN", "AAPL", "META", "MSFT", "NVDA", "TSLA"]

n_simulations = 1
results = []

for _ in range(n_simulations):
    weights = np.random.dirichlet(np.ones(7))
    portfolio_return = (normalized[mag7] * weights).sum(axis=1)
    results.append(portfolio_return)

simulations = pd.DataFrame(results).T
simulations.index = normalized.index

# One average line
normalized["IRL-M7"] = simulations.mean(axis=1)
normalized.head()

Ticker,AAPL,AMZN,GOOGL,META,MSFT,NVDA,SPY,TSLA,M7,IRL-M7
Date,,,,,,,,,,
2012-05-18,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-05-21,5.825987,1.992053,2.283471,-10.986162,1.639918,1.738376,1.718747,4.390441,0.983441,0.593178
2012-05-22,5.013389,0.692077,0.066620,-18.911855,1.674116,0.496699,1.896054,11.756175,0.112460,-1.180253
2012-05-23,7.575675,1.603930,1.508999,-16.296104,-0.546587,2.980150,1.950005,12.554454,1.340074,-0.304067
2012-05-24,6.587726,0.649988,0.542964,-13.601898,-0.683288,0.248333,2.150402,9.869415,0.516177,-0.805641


In [42]:
# Compare M7 average vs SPY
import plotly.express as px
import plotly.graph_objects as go
fig1 = px.line(normalized,x=normalized.index,y=["M7","SPY"])
fig1.update_layout(title=dict(text="M7 vs SPY",font=dict(size=24)),yaxis_title="Change(%)")
fig1.show()


In [ ]:
# compare IRL-M7 vs SPY
fig2= px.line(normalized,x=normalized.index,y=["SPY","IRL-M7"])

fig2.show()

In [36]:
#compare IRL-M7 vs avg-M7
fig3 = px.line(normalized, x=normalized.index, y=["IRL-M7", "M7"])

fig3.update_traces(selector=dict(name="IRL-M7"), line=dict(color="orange", width=2))
fig3.update_traces(selector=dict(name="M7"), line=dict(color="blue", width=2))

fig3.show()



In [37]:
fig4 = px.line(normalized,x=normalized.index,y=["GOOGL", "AMZN", "AAPL", "META", "MSFT","NVDA","TSLA"])
fig4.update_traces(opacity=0.7)
fig4.update_layout(title=dict(text="The magnificent seven's growth over-time",font=dict(size=24)))
fig4.show()

In [41]:
# Resample to monthly to make animation smoother and faster
normalized_yearly = normalized.resample("YE").last()

# Reshape to long format
normalized_long = normalized_yearly.reset_index().melt(
    id_vars="Date",
    var_name="Company",
    value_name="Return"
)

normalized_long["Date"] = normalized_long["Date"].dt.strftime("%Y-%m")

# Animate
fig5 = px.bar(normalized_long,
              x="Company",
              y="Return",
              animation_frame="Date",
              range_y=[0.01, normalized_long["Return"].max()],
              log_y=True,
              title="companies returns from 2012"
              )
fig5.show()

part 2

In [20]:
# getting the top 7 companies in 2012

In [21]:
top7_2012 = ["AAPL", "XOM", "MSFT", "IBM", "GE", "CVX", "BRK-B"]
data_2012 = yf.download(top7_2012, start="2012-05-18", end="2026-09-16")["Close"]
print(data_2012.head())

[*********************100%***********************]  7 of 7 completed

Ticker           AAPL      BRK-B        CVX         GE         IBM       MSFT  \
Date                                                                            
2012-05-18  15.863472  78.910004  54.924259  70.383820  110.360802  23.079256   
2012-05-21  16.787676  79.800003  55.610420  71.015213  111.420059  23.457737   
2012-05-22  16.658770  79.650002  55.404007  71.238052  110.890419  23.465630   
2012-05-23  17.065237  79.750000  55.225506  71.238052  110.496025  22.953108   
2012-05-24  16.908514  79.800003  55.816818  71.498055  110.479111  22.921558   

Ticker            XOM  
Date                   
2012-05-18  46.572113  
2012-05-21  46.897942  
2012-05-22  46.846481  
2012-05-23  46.897942  
2012-05-24  47.223789  


In [22]:
normalized_1= ((data_2012/data_2012.iloc[0])-1)*100
normalized_1["M7"]= normalized_1[top7_2012].mean(axis= 1)
normalized_1.head()

Ticker,AAPL,BRK-B,CVX,GE,IBM,MSFT,XOM,M7
Date,,,,,,,,
2012-05-18,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2012-05-21,5.825987,1.127866,1.249286,0.897072,0.959813,1.639918,0.699622,1.771366
2012-05-22,5.013389,0.937774,0.873472,1.213678,0.479896,1.674116,0.589126,1.540207
2012-05-23,7.575675,1.064499,0.548476,1.213678,0.122528,-0.546587,0.699622,1.525413
2012-05-24,6.587726,1.127866,1.625073,1.583084,0.107202,-0.683288,1.399284,1.678135


In [23]:
fig=go.Figure()
#first dataset(SPY)
fig.add_scatter(x=normalized.index,y=normalized["SPY"],name="SPY",mode="lines")
#second dataset(past M7)
fig.add_scatter(x=normalized_1.index,y=normalized_1["M7"],name="past M7",mode="lines")

fig.show()


In [24]:
print("Current M7:", normalized["M7"].iloc[-1].round(2), "%")
print("2012 Top 7:", normalized_1["M7"].iloc[-1].round(2), "%")
print("SPY:", normalized["SPY"].iloc[-1].round(2), "%")

Current M7: 15166.71 %
2012 Top 7: 802.7 %
SPY: 648.12 %


part 3

In [25]:
# Which company has the best run from the moment it went public

In [26]:
individual_growth={}
for ticker in mag7:
    data=yf.download(ticker,period="max")["Close"].squeeze().dropna()
    normalized_ticker= ((data/data.iloc[0])-1 )*100
    individual_growth[ticker]= normalized_ticker
df_individual=pd.DataFrame(individual_growth)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [27]:
final_returns = df_individual.iloc[-1]

fig0 = px.bar(x=final_returns.index, y=final_returns.values,
              title="Total Return Since IPO",
              labels={"x": "Company", "y": "% Return"},
              log_y=True)
fig0.show()

In [28]:
final_returns = df_individual.iloc[-1]

# Calculate years public
years = df_individual.apply(lambda x: x.dropna().shape[0] / 252).round(1)

fig0 = px.bar(x=final_returns.index, y=final_returns.values,
              title="Total Return Since IPO",
              labels={"x": "Company", "y": "% Return"},
              log_y=True,
              text=[f"{y} yrs" for y in years])

fig0.update_traces(textposition="outside")
fig0.show()

Part 4

In [30]:
growth_velocity = ((1 + final_returns/100) ** (1/years) - 1) * 100
fig_velocity=px.bar(growth_velocity,x=growth_velocity.index,y=growth_velocity.values)
fig_velocity.show()